In [ ]:
# file paths
import glob

def get_path():
  list_over_1000 = glob.glob('input/reports.disciplinary.cardTrackers*.xlsx')
  list_under_1000 = glob.glob('input/AccumulatedCards_*.xlsx')
  if len(list_over_1000) > 0:
    return list_over_1000[0]
  else:
    return list_under_1000[0]

accumulation_path = get_path()

In [ ]:
import pandas as pd
from datetime import datetime

today = datetime.today().strftime('%Y-%m-%d')

In [ ]:
yc_accumulation = pd.read_excel(accumulation_path)

yc_accumulation = yc_accumulation[yc_accumulation['Competition type'].str.contains('OPL Cup|Ontario')]

yc_accumulation['yc count'] = yc_accumulation.groupby(['Person name', 'Club', 'Type', 'Competition'])['Person name'].transform('count')

yc_accumulation = yc_accumulation[['Person name', 'Club', 'Type', 'Competition', 'yc count']]

yc_accumulation = yc_accumulation.drop_duplicates(subset=['Person name', 'Club', 'Type', 'Competition'], keep='last')

yc_accumulation = yc_accumulation[(
  ((yc_accumulation['Type'] == 'Team official') & (yc_accumulation['yc count'] >= 2)) |
  ((yc_accumulation['Type'] == 'Player') & (yc_accumulation['yc count'] == 4)) | 
  ((yc_accumulation['Type'] == 'Player') & (yc_accumulation['yc count'] >= 6))
)]

last_updated = pd.DataFrame({ 'Person name': 'Last updated:', 'Club': [today]})

yc_accumulation = pd.concat([yc_accumulation, last_updated], ignore_index=True)

yc_accumulation = yc_accumulation.rename(columns={ 'Person name': 'Name', 'Type': 'Person type' })

yc_accumulation = yc_accumulation[['Name', 'Club', 'Person type', 'Competition']]

yc_accumulation.to_csv('output/opl_yc_accumulation.csv', index=False)

yc_accumulation

